# Forest Gain Inspection Notebook
Derived from `gee.py`. No export behaviour — visualisation and tile-level statistics only.

In [ ]:
!pip install geemap earthengine-api ipyleaflet ipywidgets -q

In [ ]:
import ee
import geemap
import ipywidgets as widgets

ee.Authenticate()
ee.Initialize()

## Configuration
Edit `BOUNDS` to change the AOI. Everything downstream recomputes automatically.

In [ ]:
BOUNDS = [-4.04, 51.54, -3.32, 51.75]  # [min_lon, min_lat, max_lon, max_lat]

TILE_PIXELS = 128
SCALE = 10
TILE_METERS = TILE_PIXELS * SCALE

## AOI — snapped to tile grid

In [ ]:
def build_aoi(bounds):
    """Snap a bounding box to the nearest tile-aligned coordinates."""
    raw = ee.Geometry.Rectangle(bounds)
    coords = ee.List(raw.coordinates().get(0))

    raw_min_lon = ee.Number(ee.List(coords.get(0)).get(0))
    raw_min_lat = ee.Number(ee.List(coords.get(0)).get(1))
    raw_max_lon = ee.Number(ee.List(coords.get(2)).get(0))
    raw_max_lat = ee.Number(ee.List(coords.get(2)).get(1))

    tile_deg_lat = ee.Number(TILE_METERS).divide(111320)
    center_lat = raw_min_lat.add(raw_max_lat).divide(2)
    lat_cos = center_lat.multiply(3.141592653589793 / 180).cos()
    tile_deg_lon = ee.Number(TILE_METERS).divide(111320).divide(lat_cos)

    min_lon = raw_min_lon.divide(tile_deg_lon).floor().multiply(tile_deg_lon)
    min_lat = raw_min_lat.divide(tile_deg_lat).floor().multiply(tile_deg_lat)
    max_lon = raw_max_lon.divide(tile_deg_lon).ceil().multiply(tile_deg_lon)
    max_lat = raw_max_lat.divide(tile_deg_lat).ceil().multiply(tile_deg_lat)

    aoi = ee.Geometry.Rectangle([min_lon, min_lat, max_lon, max_lat])
    return aoi, min_lon, min_lat, max_lon, max_lat, tile_deg_lon, tile_deg_lat


aoi, min_lon, min_lat, max_lon, max_lat, tile_deg_lon, tile_deg_lat = build_aoi(BOUNDS)
print("AOI built and snapped to tile grid.")

## Gain layer

In [ ]:
def build_gain_layer(aoi):
    worldcover = ee.Image("ESA/WorldCover/v100/2020").clip(aoi)
    is_forest = worldcover.eq(10)

    m15 = ee.Image("projects/glad/GLCLU2020/v2/LCLUC_2015").clip(aoi)
    m20 = ee.Image("projects/glad/GLCLU2020/v2/LCLUC_2020").clip(aoi)

    tree_classes = ee.List.sequence(25, 96).cat(ee.List.sequence(125, 196))
    ones = ee.List.repeat(1, tree_classes.length())

    tree2015 = m15.remap(tree_classes, ones, 0)
    tree2020 = m20.remap(tree_classes, ones, 0)

    forest_gain = tree2020.And(tree2015.Not())
    clean_gain = forest_gain.updateMask(forest_gain).focal_max(1).focal_min(1)
    gain_validated = clean_gain.And(is_forest)
    gain_binary = gain_validated.unmask(0).rename("gain")

    return gain_validated, gain_binary, is_forest


gain_validated, gain_binary, is_forest = build_gain_layer(aoi)
print("Gain layer built.")

## S2 data validity mask

In [ ]:
def s2_availability(aoi, year):
    start = "2015-01-01" if year == 2016 else f"{year}-01-01"
    end = "2016-12-31" if year == 2016 else f"{year}-12-31"
    ic = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(start, end)
        .filterBounds(aoi)
        .select(["B2", "B3", "B4", "B5", "B6", "B7", "B8"])
    )
    return ic.map(lambda img: img.mask().reduce(ee.Reducer.min())).reduce(ee.Reducer.max())


full_valid = (
    s2_availability(aoi, 2016)
    .And(s2_availability(aoi, 2020))
    .And(s2_availability(aoi, 2025))
    .selfMask()
    .rename("valid")
)
print("Validity mask built.")

## Ancillary layers — DEM, canopy, JRC, natural forest

In [ ]:
fabdem = (
    ee.ImageCollection("projects/sat-io/open-datasets/FABDEM")
    .filterBounds(aoi)
    .mosaic()
    .clip(aoi)
)
slope = ee.Terrain.slope(fabdem)

canopy_raw = ee.Image("users/nlang/ETH_GlobalCanopyHeight_2020_10m_v1")
canopy = canopy_raw.clip(aoi).rename("canopy_height").updateMask(canopy_raw.gte(0))
gain_height = canopy.updateMask(gain_validated).rename("canopy_gain_height")

jrc = ee.Image("JRC/GFC2020_subtypes/V1").clip(aoi).rename("jrc_forest_type")

nat_forest = (
    ee.ImageCollection(
        "projects/nature-trace/assets/forest_typology/natural_forest_2020_v1_0_collection"
    )
    .mosaic()
    .select("B0")
    .divide(250)
    .clip(aoi)
    .unmask(0)
    .rename("natural_forest_prob")
)
print("Ancillary layers built.")

## Tile grid and per-tile statistics
Each tile gets: mean gain fraction, mean canopy height, modal JRC forest type, mean natural forest probability, and mean slope. Tiles are filtered to ≥1% gain coverage with full S2 validity.

In [ ]:
def build_grid(min_lon, min_lat, max_lon, max_lat, tile_deg_lon, tile_deg_lat):
    def make_tiles(lon):
        lon = ee.Number(lon)

        def inner(lat):
            lat = ee.Number(lat)
            tile_id = (
                ee.String("tile_")
                .cat(lon.multiply(1e6).round().format("%d"))
                .cat("_")
                .cat(lat.multiply(1e6).round().format("%d"))
            )
            return ee.Feature(
                ee.Geometry.Rectangle([lon, lat, lon.add(tile_deg_lon), lat.add(tile_deg_lat)]),
                {"tile_id": tile_id},
            )

        return ee.List.sequence(min_lat, max_lat, tile_deg_lat).map(inner)

    return ee.FeatureCollection(
        ee.List.sequence(min_lon, max_lon, tile_deg_lon).map(make_tiles).flatten()
    )


grid = build_grid(min_lon, min_lat, max_lon, max_lat, tile_deg_lon, tile_deg_lat)

# Stack all continuous metrics into one image for a single reduceRegions call
metrics_img = ee.Image.cat([
    gain_binary.rename("gain"),
    gain_height.unmask(0).rename("canopy"),
    nat_forest.rename("nat_forest"),
    slope.rename("slope"),
    full_valid.unmask(0).rename("valid"),
])

tile_means = metrics_img.reduceRegions(
    collection=grid,
    reducer=ee.Reducer.mean(),
    scale=SCALE,
    tileScale=4,
)

# JRC needs a separate mode reduction
tile_jrc = jrc.reduceRegions(
    collection=grid,
    reducer=ee.Reducer.mode(),
    scale=SCALE,
    tileScale=4,
)

# Join JRC mode onto the mean stats
join = ee.Join.saveFirst("jrc_match")
filter = ee.Filter.equals(leftField="system:index", rightField="system:index")
joined = join.apply(tile_means, tile_jrc, filter)

def merge_jrc(f):
    jrc_mode = ee.Feature(f.get("jrc_match")).get("mode")
    gain_pct = ee.Number(f.get("gain")).multiply(100)
    return f.set({"jrc_mode": jrc_mode, "gain_pct": gain_pct})

tiles_with_stats = (
    joined
    .map(merge_jrc)
    .filter(ee.Filter.gte("gain", 0.01))   # >= 1% gain coverage
    .filter(ee.Filter.eq("valid", 1))       # full S2 validity
)

print("Tile grid and statistics built.")

## Sentinel-2 composites for display

In [ ]:
def add_indices(img):
    ndvi = img.normalizedDifference(["B8", "B4"]).rename("NDVI")
    evi = img.expression(
        "2.5 * ((NIR - RED) / (NIR + 6.0 * RED - 7.5 * BLUE + 1.0))",
        {"NIR": img.select("B8"), "RED": img.select("B4"), "BLUE": img.select("B2")},
    ).rename("EVI")
    return img.select(["B2", "B3", "B4", "B5", "B6", "B7", "B8"]).addBands([ndvi, evi])


def s2_composite(aoi, year):
    start = "2015-01-01" if year == 2016 else f"{year}-01-01"
    end = "2016-12-31" if year == 2016 else f"{year}-12-31"
    return (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(start, end)
        .filterBounds(aoi)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 30))
        .map(add_indices)
        .median()
    )


s2_2016 = s2_composite(aoi, 2016)
s2_2020 = s2_composite(aoi, 2020)
s2_2025 = s2_composite(aoi, 2025)
print("S2 composites built.")

## Map
Click any green tile to see its statistics in the panel (bottom-right).

In [ ]:
Map = geemap.Map(center=[51.6, -3.7], zoom=10)

# Sentinel RGB
rgb_vis = {"min": 200, "max": 3000, "gamma": 1.3}
Map.addLayer(s2_2016.clip(aoi).select(["B4", "B3", "B2"]), rgb_vis, "S2 2016", shown=False)
Map.addLayer(s2_2020.clip(aoi).select(["B4", "B3", "B2"]), rgb_vis, "S2 2020")
Map.addLayer(s2_2025.clip(aoi).select(["B4", "B3", "B2"]), rgb_vis, "S2 2025", shown=False)

# Gain
Map.addLayer(gain_validated.selfMask(), {"palette": ["00E676"]}, "Gain validated")

# Forest cover
Map.addLayer(is_forest.selfMask(), {"palette": ["006400"]}, "Forest 2020", shown=False)

# Canopy height at gain pixels
Map.addLayer(
    gain_height,
    {"min": 0, "max": 40, "palette": ["f7fcf5", "c7e9c0", "74c476", "238b45", "00441b"]},
    "Canopy gain height",
    shown=False,
)

# JRC forest type
Map.addLayer(
    jrc,
    {"min": 1, "max": 20, "palette": ["78c679", "006837", "cc6600"]},
    "JRC forest type",
    shown=False,
)

# Natural forest probability
Map.addLayer(
    nat_forest,
    {"min": 0, "max": 1, "palette": ["white", "green"]},
    "Natural forest prob",
    shown=False,
)

# Filtered tile outlines (visual layer — styled, no properties)
Map.addLayer(
    tiles_with_stats.style(color="00FF00", fillColor="00000000", width=2),
    {},
    "Filtered tiles",
)

Map.addLayerControl()

# --- Click handler: query tiles_with_stats directly ---
output = widgets.Output(
    layout=widgets.Layout(
        width="280px",
        max_height="220px",
        overflow_y="auto",
        border="1px solid #ccc",
        padding="8px",
        background_color="white",
    )
)

with output:
    print("Click a tile to see statistics.")


def handle_click(**kwargs):
    if kwargs.get("type") != "click":
        return
    coords = kwargs["coordinates"]
    point = ee.Geometry.Point([coords[1], coords[0]])
    info = tiles_with_stats.filterBounds(point).first().getInfo()
    with output:
        output.clear_output()
        if not info:
            print("No filtered tile at this location.")
            return
        p = info["properties"]

        def fmt(v, decimals=2):
            return f"{float(v):.{decimals}f}" if v is not None else "N/A"

        print(f"Tile ID     : {p.get('tile_id', 'N/A')}")
        print(f"Gain %      : {fmt(p.get('gain_pct'))}%")
        print(f"Canopy (m)  : {fmt(p.get('canopy'))}")
        print(f"Nat forest  : {fmt(p.get('nat_forest'))}")
        print(f"Slope (°)   : {fmt(p.get('slope'))}")
        print(f"JRC mode    : {p.get('jrc_mode', 'N/A')}")


Map.on_interaction(handle_click)
Map.add_widget(output, position="bottomright")

Map